# M3 微調一鍵比較(nano + medium × 有無微調)

**這份筆記本從頭到尾自動跑完,不用手動改模型。** 一次產出:
- **nano** 與 **medium** 兩個變體,各自「**微調前(COCO 原始)**」vs「**微調後**」在 test 集的偵測數據。
- 訓練過程會自動印出每個變體的 **per-class AP 表**(val)。
- 最後印出總結表,並下載 `m3_compare_results.json` + 兩個微調權重。

## 用法(三步)
1. `執行階段 → 變更執行階段類型 → 硬體加速器:GPU(T4)`。
2. `執行階段 → 全部執行`。
3. 跳出選檔時上傳你的 `dataset.zip`。

⚠ 兩個模型都要訓練,總共約 **60~90 分鐘**,請保持分頁開著、別關電腦。
⚠(bootstrap)EPFL 為 CC-NC,僅驗證用、不出貨;真實資料照同一份跑即可。

In [ ]:
# 1) 檢查 GPU + 安裝(train,loggers 才含訓練依賴 pytorch_lightning)
!nvidia-smi -L
!pip -q install "rfdetr[train,loggers]" supervision

In [ ]:
# 2) 上傳 dataset.zip 並解壓(跳出選檔時選你的 dataset.zip)
from google.colab import files
up = files.upload()
!unzip -q -o dataset.zip -d /content/m3ds
!ls /content/m3ds   # 應看到 train valid test

In [ ]:
# 3) 設定 + 評估工具(不用改)
import os, json, gc
from PIL import Image
try:
    import torch
except Exception:
    torch = None

TEST = '/content/m3ds/test'
NAMES = {0:'人',1:'刀具',2:'砧板',3:'食材',4:'鍋鏟',5:'鍋子',6:'手',7:'容器',8:'抹布',9:'夾子',10:'手套'}
# COCO 類別 id(1..11) → 模型類別 id(0..10)
CAT2CLS = {i: i-1 for i in range(1, 12)}

def free():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def _load_gt(test_dir):
    j = json.load(open(os.path.join(test_dir, '_annotations.coco.json'), encoding='utf-8'))
    id2fn = {im['id']: im['file_name'] for im in j['images']}
    by_img = {}
    for a in j['annotations']:
        x, y, w, h = a['bbox']
        by_img.setdefault(id2fn[a['image_id']], []).append((a['category_id'], [x, y, x+w, y+h]))
    return by_img

def eval_model(model, test_dir, cat_to_class=None, thr=0.3, iou_thr=0.5):
    '''cat_to_class=None → 只算有沒有框到物件(微調前用);給對照表 → 也算類別對不對。'''
    gt = _load_gt(test_dir)
    per = {}
    for fn, objs in gt.items():
        det = model.predict(Image.open(os.path.join(test_dir, fn)).convert('RGB'), threshold=thr)
        pboxes = det.xyxy.tolist() if len(det) else []
        pcls = [int(c) for c in det.class_id] if len(det) else []
        used = [False] * len(pboxes)
        for gcat, gbox in objs:
            per.setdefault(gcat, [0, 0, 0])
            per[gcat][0] += 1
            best, bestv = -1, iou_thr
            for i, pb in enumerate(pboxes):
                if used[i]:
                    continue
                v = _iou(gbox, pb)
                if v >= bestv:
                    bestv, best = v, i
            if best >= 0:
                used[best] = True
                per[gcat][1] += 1
                if cat_to_class is not None and pcls[best] == cat_to_class.get(gcat, -999):
                    per[gcat][2] += 1
    n_gt = sum(p[0] for p in per.values())
    n_loc = sum(p[1] for p in per.values())
    n_cls = sum(p[2] for p in per.values())
    out = {'overall': {'n_gt': n_gt,
                       'recall_localize': round(n_loc/max(1, n_gt), 3),
                       'recall_correct': (round(n_cls/max(1, n_gt), 3) if cat_to_class is not None else None)},
           'per_class': {}}
    for gcat in sorted(per):
        g, l, c = per[gcat]
        out['per_class'][NAMES.get(gcat-1, gcat)] = {
            'n_gt': g,
            'recall_localize': round(l/max(1, g), 3),
            'recall_correct': (round(c/max(1, g), 3) if cat_to_class is not None else None)}
    return out

print('工具就緒。test 圖數:', len(_load_gt(TEST)))

In [ ]:
# 4) 自動跑兩個變體:微調前 → 微調 → 微調後(約 60~90 分鐘)
from rfdetr import RFDETRNano, RFDETRMedium

VARIANTS = [
    {'name': 'nano',   'cls': RFDETRNano,   'batch': 4, 'out': '/content/out_nano'},
    {'name': 'medium', 'cls': RFDETRMedium, 'batch': 2, 'out': '/content/out_medium'},
]

results = {}
for v in VARIANTS:
    print()
    print('=' * 64)
    print('變體:', v['name'])
    print('=' * 64)

    # (a) 微調前:COCO 原始權重。類別對不上我們11類,只看有沒有框到物件
    base = v['cls']()
    before = eval_model(base, TEST, cat_to_class=None)
    print(v['name'], '微調前 →', before['overall'])
    base = None
    free()

    # (b) 微調(過程會自動印每個 epoch 的 per-class AP 表)
    model = v['cls']()
    model.train(dataset_dir='/content/m3ds', epochs=60, batch_size=v['batch'],
                grad_accum_steps=4, lr=1e-4, resolution=704, output_dir=v['out'])
    model = None
    free()

    # (c) 微調後:載入權重(不要 optimize),類別對得上,算定位+分類正確
    ckpt = v['out'] + '/checkpoint_best_regular.pth'
    ft = v['cls'](pretrain_weights=ckpt, num_classes=11)
    after = eval_model(ft, TEST, cat_to_class=CAT2CLS)
    print(v['name'], '微調後 →', after['overall'])
    ft = None
    free()

    results[v['name']] = {'before': before, 'after': after, 'ckpt': ckpt}

print()
print('全部跑完 ✅')

In [ ]:
# 5) 總結表 + 存檔下載
print('======== 微調前 vs 微調後(test 集,IoU>=0.5)========')
print()
print('{:<8}{:>16}{:>16}{:>18}'.format('變體', '前_定位Recall', '後_定位Recall', '後_分類正確Recall'))
print('-' * 58)
for name, r in results.items():
    b = r['before']['overall']
    a = r['after']['overall']
    print('{:<8}{:>16}{:>16}{:>18}'.format(name, b['recall_localize'], a['recall_localize'], a['recall_correct']))

print()
print('---- 微調後 各類別 分類正確Recall ----')
for name, r in results.items():
    print()
    print('[' + name + ']')
    for cls, m in r['after']['per_class'].items():
        print('  {:<4} n={:<3} 正確Recall={}'.format(cls, m['n_gt'], m['recall_correct']))

json.dump(results, open('/content/m3_compare_results.json', 'w'), ensure_ascii=False, indent=2)
print()
print('已存 /content/m3_compare_results.json')

from google.colab import files
files.download('/content/m3_compare_results.json')
for name, r in results.items():
    try:
        files.download(r['ckpt'])
    except Exception as e:
        print('下載', name, '權重失敗(可稍後手動下載):', e)

## 怎麼看這些數據

| 指標 | 意思 |
|---|---|
| **定位 Recall(微調前)** | COCO 原始模型有沒有框到物件。它不認得我們的11類,只能看有無框到 → 通常很低,證明現成模型不夠用。 |
| **定位 Recall(微調後)** | 微調後有沒有框到 → 應明顯上升。 |
| **分類正確 Recall(微調後)** | 框到**且類別也對** → 這才是真正可用的指標,食安關鍵類別(刀具/砧板/食材/手)越高越好。 |

> **更細的 per-class AP** 在上面**第 4 格訓練過程**會自動印出(rfdetr 每個變體各印一張 val 表);本頁總結是 test 集的 Recall,兩者互補。

## 要交回的東西
1. `m3_compare_results.json`(本頁自動下載)。
2. 第 4 格印出的 **nano、medium 兩張 per-class AP 表**(截圖)。
3. 兩個權重 `checkpoint_best_regular.pth`(nano/medium,本頁自動下載)。